# 02 · Análisis con SQL

**Bloque 2 de 4 · 35 minutos**

En el bloque anterior dejamos los datos limpios. Ahora vamos a responder la
primera pregunta del directivo — **¿cuánto se disparó?** — construyendo la
consulta paso a paso.

> 🛟 **Si no terminaste el bloque 1, no pasa nada.** Este notebook arranca de
> los archivos ya preparados en `datos/checkpoints/`.

---
## 📖 2.0 · Cargar los datos y hacer la primera consulta

DuckDB tiene una propiedad muy cómoda: **puede consultar tus DataFrames de
pandas directamente**, sin cargarlos en ninguna base de datos. Basta con que
la variable exista.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from utils.verificar import verificar

CK = RAIZ / "datos" / "checkpoints"

hechos = pd.read_parquet(CK / "hechos_limpios.parquet")
presupuesto = pd.read_parquet(CK / "presupuesto_limpio.parquet")
dim_cc = pd.read_parquet(CK / "dim_centro_costo_limpio.parquet")

# Los tipos vienen bien porque Parquet los conserva: no hay que repetir
# la limpieza del bloque 1.
print(f"hechos       {len(hechos):>7,} filas   monto es {hechos['monto'].dtype}")
print(f"presupuesto  {len(presupuesto):>7,} filas")
print(f"dim_cc       {len(dim_cc):>7,} filas")

hechos        12,000 filas   monto es float64
presupuesto      576 filas
dim_cc             8 filas


In [ ]:
# Necesitamos año y mes por separado para poder agrupar por periodo
hechos["anio"] = hechos["fecha"].str[:4].astype(int)
hechos["mes"] = hechos["fecha"].str[5:7].astype(int)

# Primera consulta: DuckDB ve la variable `hechos` directamente
duckdb.sql("""
    SELECT planta, COUNT(*) AS asientos, SUM(monto) AS gasto
    FROM hechos
    GROUP BY planta
    ORDER BY gasto DESC
""").df()

,planta,asientos,gasto
0,Chihuahua,2880,15240000.0
1,Hermosillo,3320,12900000.0
2,Cuautitlán,2920,12300000.0
3,Irapuato,2880,9870000.0


> ⚠️ **Cuidado con los nombres de columna.** `real` es una palabra reservada
> en SQL (es un tipo de dato), así que si escribes `SUM(monto) AS real` la
> consulta truena. Por eso usamos `gasto_real`.

### Tabla de traducción · pandas ↔ SQL

Ya sabes hacer todo esto en pandas. Son la misma idea escrita de dos formas:

| pandas | SQL |
|---|---|
| `df[df.planta == "Hermosillo"]` | `WHERE planta = 'Hermosillo'` |
| `df.groupby("centro_costo")["monto"].sum()` | `GROUP BY centro_costo` + `SUM(monto)` |
| `df.merge(otro, on="centro_costo")` | `JOIN otro ON a.centro_costo = b.centro_costo` |
| `df.sort_values("monto", ascending=False)` | `ORDER BY monto DESC` |
| `df.head(5)` | `LIMIT 5` |
| `df.groupby(...).agg(...)` sobre grupos | `GROUP BY` con varias columnas |

**Usa la que te resulte más clara.** SQL suele ganar cuando hay varios joins
y agregaciones a la vez, que es justo nuestro caso.

---
## ✏️ 2.2 · TU TURNO · Filtrar el periodo que pidieron

El directivo preguntó por una planta y un trimestre concretos. Completa el
`WHERE`.

> 💡 **Pista:** el directivo dijo Hermosillo y Q2. El segundo trimestre son
> tres meses — revisa cuáles.

In [ ]:
gasto_q2 = duckdb.sql("""
    SELECT folio, fecha, centro_costo, monto
    FROM hechos
    WHERE planta = '____________'
      AND anio = 2026
      AND mes BETWEEN __ AND __
""").df()

print(f"{len(gasto_q2):,} asientos")
gasto_q2.head()

In [ ]:
verificar("2.2", gasto_q2)

---
## 📖 2.3 · Agrupar: de 800 asientos a 8 cifras

Un directivo no quiere 800 renglones. Quiere el total por centro de costo.

In [ ]:
duckdb.sql("""
    SELECT centro_costo,
           COUNT(*)   AS asientos,
           SUM(monto) AS gasto_real
    FROM hechos
    WHERE planta = 'Hermosillo' AND anio = 2026 AND mes BETWEEN 4 AND 6
    GROUP BY centro_costo
    ORDER BY gasto_real DESC
""").df()

,centro_costo,asientos,gasto_real
0,Mantenimiento,100,950000.0
1,Fletes,100,570000.0
2,Energía,100,470000.0
3,Nómina Indirecta,100,325000.0
4,Refacciones,100,210000.0
5,Calidad,100,180000.0
6,Seguridad,100,150000.0
7,Sistemas,100,145000.0


---
## ✏️ 2.4 · TU TURNO · Agrupar también por mes

Ocho cifras responden *¿dónde?* pero no *¿cuándo?*. Para eso necesitamos el
detalle por centro de costo **y** por mes.

> 💡 **Pista:** en SQL, todo lo que va en el `SELECT` sin una función de
> agregación tiene que ir también en el `GROUP BY`. Necesitas dos columnas,
> no una.

In [ ]:
gasto_mensual = duckdb.sql("""
    SELECT centro_costo,
           mes,
           SUM(monto) AS gasto_real
    FROM hechos
    WHERE planta = 'Hermosillo' AND anio = 2026 AND mes BETWEEN 4 AND 6
    GROUP BY ____________, ___
    ORDER BY centro_costo, mes
""").df()

print(f"{len(gasto_mensual)} filas  (8 centros × 3 meses)")
gasto_mensual.head(6)

In [ ]:
verificar("2.4", gasto_mensual)

---
## 📖 2.5 · Combinar gasto real con presupuesto

Aquí está el corazón del análisis. Tenemos dos tablas:

```
   hechos                        presupuesto
   ───────────────────────       ───────────────────────
   planta                        planta
   centro_costo                  centro_costo
   anio, mes                     anio, mes
   monto        (lo gastado)     monto_presupuesto  (lo autorizado)
```

Para comparar peras con peras, cada fila de gasto tiene que casar con **la
fila de presupuesto de la misma planta, el mismo centro de costo y el mismo
mes**. Son tres llaves.

La estrategia: agregar cada lado por separado y unir los resultados.

In [ ]:
# Así se ve la estructura antes de completarla
consulta_ejemplo = """
    WITH real AS (
        SELECT planta, centro_costo, anio, mes, SUM(monto) AS gasto_real
        FROM hechos
        GROUP BY planta, centro_costo, anio, mes
    ),
    ppto AS (
        SELECT planta, centro_costo, anio, mes,
               SUM(monto_presupuesto) AS presupuesto
        FROM presupuesto
        GROUP BY planta, centro_costo, anio, mes
    )
    SELECT * FROM real LIMIT 3
"""
duckdb.sql(consulta_ejemplo).df()

,planta,centro_costo,anio,mes,gasto_real
0,Irapuato,Nómina Indirecta,2025,1,91622.00
1,Irapuato,Fletes,2025,1,72316.50
2,Hermosillo,Calidad,2025,2,51206.75


> ℹ️ `WITH nombre AS (...)` crea una tabla temporal para esa consulta. Sirve
> para no anidar todo en una sola instrucción ilegible. En pandas equivale a
> guardar un resultado intermedio en una variable.

---
## ✏️ 2.6 · TU TURNO · Completar el JOIN

Completa el `ON` para que cada fila de gasto case con su fila de presupuesto.

> 💡 **Pista:** para comparar gasto real contra presupuesto, ambos deben
> referirse a lo mismo: la misma planta, el mismo centro de costo y el mismo
> mes. Si te falta una llave, estarás comparando peras con manzanas.

In [ ]:
comparacion = duckdb.sql("""
    WITH real AS (
        SELECT planta, centro_costo, anio, mes, SUM(monto) AS gasto_real
        FROM hechos GROUP BY planta, centro_costo, anio, mes
    ),
    ppto AS (
        SELECT planta, centro_costo, anio, mes,
               SUM(monto_presupuesto) AS presupuesto
        FROM presupuesto GROUP BY planta, centro_costo, anio, mes
    )
    SELECT r.planta, r.centro_costo, r.anio, r.mes,
           r.gasto_real, p.presupuesto
    FROM real r
    JOIN ppto p
      ON  r.planta = p.planta
      AND r.____________ = p.____________
      AND r.____ = p.____
      AND r.___ = p.___
""").df()

print(f"{len(comparacion):,} filas  (esperado: 576 = 4 plantas × 8 centros × 18 meses)")
comparacion.head()

In [ ]:
verificar("2.6", comparacion)

> 🔑 **Qué pasa si te falta una llave.** Si quitas `AND r.mes = p.mes`, cada
> mes de gasto casa con **los 18 meses** de presupuesto y el resultado se
> multiplica. El total se infla sin que nada truene. Se llama *fan-out* y es
> uno de los errores más caros en reportes financieros: los números salen
> enormes y plausibles.

---
## ✏️ 2.7 · TU TURNO · Calcular la desviación

Último paso del bloque: la cifra que el directivo pidió.

> 💡 **Pista:** la desviación es la diferencia entre lo que se gastó y lo que
> estaba autorizado, y el signo importa: **positivo debe significar
> sobregasto**. Para el porcentaje, piensa sobre qué base lo lee un
> directivo: ¿sobre lo gastado, o sobre lo que había autorizado?

In [ ]:
variance = duckdb.sql("""
    WITH real AS (
        SELECT planta, centro_costo, anio, mes, SUM(monto) AS gasto_real
        FROM hechos GROUP BY planta, centro_costo, anio, mes
    ),
    ppto AS (
        SELECT planta, centro_costo, anio, mes,
               SUM(monto_presupuesto) AS presupuesto
        FROM presupuesto GROUP BY planta, centro_costo, anio, mes
    )
    SELECT r.planta, r.centro_costo, r.anio, r.mes,
           r.gasto_real,
           p.presupuesto,
           ______________ - ______________            AS variance,
           ROUND(100.0 * (______________ - ______________)
                       / ______________, 2)           AS variance_pct
    FROM real r
    JOIN ppto p
      ON  r.planta = p.planta AND r.centro_costo = p.centro_costo
      AND r.anio = p.anio     AND r.mes = p.mes
""").df()

print(f"{len(variance):,} filas")
variance.head()

In [ ]:
verificar("2.7", variance)

---
## 📖 2.8 · La respuesta a la primera pregunta

Con la tabla de desviación ya podemos contestar *¿cuánto se disparó?*

In [ ]:
duckdb.sql("""
    SELECT planta,
           ROUND(SUM(gasto_real), 2)   AS gasto_real,
           ROUND(SUM(presupuesto), 2)  AS presupuesto,
           ROUND(SUM(variance), 2)     AS variance
    FROM variance
    WHERE anio = 2026 AND mes BETWEEN 4 AND 6
    GROUP BY planta
    ORDER BY variance DESC
""").df()

,planta,gasto_real,presupuesto,variance
0,Chihuahua,3540000.0,2340000.0,1200000.0
1,Hermosillo,3000000.0,2000000.0,1000000.0
2,Irapuato,1770000.0,1620000.0,150000.0
3,Cuautitlán,1800000.0,2100000.0,-300000.0


Guarda esa tabla. En el bloque 3 la vamos a graficar — y vas a descubrir algo
que el directivo no preguntó.

In [ ]:
variance.to_parquet(CK / "hechos_variance.parquet", index=False)
print(f"Guardado: datos/checkpoints/hechos_variance.parquet  ({len(variance):,} filas)")

Guardado: datos/checkpoints/hechos_variance.parquet  (576 filas)


---

# ✅ Bloque 2 terminado

Construiste una consulta que **filtra**, **agrupa** y **combina tres tablas**
para producir una cifra defendible.

```
   ¿Cuánto?    ✅ ya la tenemos
   ¿Cuándo?    ⬜ bloque 3
   ¿Dónde?     ⬜ bloque 3
```

---

## 🎯 Ahora ve al reto — Paso 2

Abre **`04_reto_final.ipynb`** y haz el **Paso 2** (5 minutos).

Tu consulta de desviación sirve casi tal cual. Solo cambia la planta.

Después: **☕ break de 20 minutos**, y seguimos con
`03_storytelling_y_salida.ipynb`.